# Preprocessing Pipeline
## Postoperative Complications After Esophageal Cancer Surgery -- DUCA Registry 2016-2025

---

This notebook continues directly from the **Descriptive Statistics notebook**.
It implements all preprocessing actions identified in the readiness analysis:

1. Load the merged dataset (or re-run merge if needed)
2. Standardise missing value codes (##USER_MISSING_99##, Onbekend, etc.)
3. Drop variables with >50% missing or very low prevalence (<2%)
4. Encode categorical and ordinal variables
5. Impute missing values
6. Log-transform skewed continuous variables
7. Final check and export of clean dataset

**At the end of this notebook you will have a clean, model-ready dataframe.**


---
## 1. Load Data

We start from the merged and cleaned dataframe.
If you are running this notebook in a fresh session, re-run the merge steps below.
Otherwise, skip to Section 2 if `df` is already in memory.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Source data is not included in this repository (governed by DUCA/AUMC data agreement) — point this at your own extract.
FILE_PATH = "../data/duca_esophagectomy_raw.xlsx"


In [ ]:
# ── Run this block if df is not already in memory ──
try:
    _ = df.shape
    print(f"df already loaded: {df.shape[0]} rows, {df.shape[1]} columns -- skipping reload")
except NameError:
    print("df not found -- loading and merging now...")

    df1 = pd.read_excel(FILE_PATH, sheet_name=0)
    df2 = pd.read_excel(FILE_PATH, sheet_name=1)

    def pad_upn(df, col='upn', digits=7):
        df = df.copy()
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].str.split('.').str[0]
        df[col] = df[col].str.zfill(digits)
        return df

    df1 = pad_upn(df1)
    df2 = pad_upn(df2)

    df = df1.merge(df2, on='upn', how='left')

    x_cols = [c for c in df.columns if c.endswith('_x')]
    y_cols = [c for c in df.columns if c.endswith('_y')]
    df = df.drop(columns=y_cols)
    df = df.rename(columns={c: c[:-2] for c in x_cols})

    # Compute age_at_op
    if 'datok' in df.columns and 'gebdat' in df.columns:
        datok_dt  = pd.to_datetime(df['datok'],  errors='coerce', dayfirst=True)
        gebdat_dt = pd.to_datetime(df['gebdat'], errors='coerce', dayfirst=True)
        if datok_dt.isna().mean() > 0.5:
            datok_dt  = pd.to_datetime(df['datok'],  unit='D', origin='1899-12-30', errors='coerce')
        if gebdat_dt.isna().mean() > 0.5:
            gebdat_dt = pd.to_datetime(df['gebdat'], unit='D', origin='1899-12-30', errors='coerce')
        df['age_at_op'] = (datok_dt - gebdat_dt).dt.days // 365

    # Compute BMI
    if 'BMI' not in df.columns and 'gewicht' in df.columns and 'lengte' in df.columns:
        df['BMI'] = pd.to_numeric(df['gewicht'], errors='coerce') /                     (pd.to_numeric(df['lengte'], errors='coerce') / 100) ** 2

    print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")


In [ ]:
# ── Patient Cohort Flowchart ──────────────────────────────────────────────────
import matplotlib.patches as mpatches

# Cohort counts at each step
n_merged  = len(df)
n_miss_oc = int(df['compl'].isna().sum()) if 'compl' in df.columns else 0
n_analysis = n_merged - n_miss_oc

try:
    n_sheet1 = len(df1)
except NameError:
    n_sheet1 = n_merged   # df1 not in memory — show merged count only

# ── draw ──
fig, ax = plt.subplots(figsize=(5, 10))
ax.set_xlim(0, 10)
ax.set_ylim(0, 22)
ax.axis('off')

def fc_box(ax, cx, cy, w, h, txt, color='#DAE8FC'):
    rect = mpatches.FancyBboxPatch(
        (cx - w/2, cy - h/2), w, h,
        boxstyle='round,pad=0.15',
        facecolor=color, edgecolor='#2C5F8A', linewidth=1.8)
    ax.add_patch(rect)
    ax.text(cx, cy, txt, ha='center', va='center',
            fontsize=10, fontweight='bold', multialignment='center')

def fc_arrow(ax, cx, y_from, y_to):
    ax.annotate('', xy=(cx, y_to + 0.45), xytext=(cx, y_from - 0.45),
                arrowprops=dict(arrowstyle='->', color='#2C5F8A', lw=2.0))

def fc_excl(ax, cx, mid_y, txt):
    ax.annotate('', xy=(cx + 3.2, mid_y), xytext=(cx + 0.55, mid_y),
                arrowprops=dict(arrowstyle='->', color='#C0392B', lw=1.5))
    ax.text(cx + 3.3, mid_y, txt, ha='left', va='center',
            fontsize=9, color='#C0392B', style='italic')

rows = [19, 13.5, 8]
fc_box(ax, 5, rows[0], 7.5, 2.2,
       f'DUCA Registry\n2016\u20132025\nn = {n_sheet1}', '#D5E8D4')
fc_arrow(ax, 5, rows[0], rows[1])

fc_box(ax, 5, rows[1], 7.5, 2.2,
       f'After ID linkage & sheet merge\nn = {n_merged}', '#DAE8FC')
fc_arrow(ax, 5, rows[1], rows[2])

if n_miss_oc > 0:
    fc_excl(ax, 5, (rows[1] + rows[2]) / 2,
            f'Missing outcome\nexcluded (n = {n_miss_oc})')

fc_box(ax, 5, rows[2], 7.5, 2.2,
       f'Analysis cohort\nn = {n_analysis}', '#D5E8D4')

ax.set_title('Patient Inclusion Flowchart', fontweight='bold', fontsize=13, pad=6)
plt.tight_layout()
plt.show()
print(f"\nRegistry: {n_sheet1}  ->  merged: {n_merged}  ->  analysis: {n_analysis}  (excluded {n_miss_oc} missing outcome)")


---
## 2. Standardise Missing Value Codes

Some columns use special codes for missing values instead of NaN:
- `##USER_MISSING_99##` -- system missing code
- `Onbekend` -- Dutch for "unknown"
- `9`, `99`, `999` -- numeric sentinel codes for unknown

We replace all of these with proper `NaN` so pandas handles them correctly.


In [ ]:
# Replace all known missing value codes with NaN
MISSING_CODES = ['##USER_MISSING_99##', 'Onbekend', 'onbekend',
                 'UNKNOWN', 'unknown', 'NA', 'N/A', 'n/a', '-']

for col in df.columns:
    # Replace string missing codes
    if df[col].dtype == object:
        df[col] = df[col].replace(MISSING_CODES, np.nan)

    # Replace numeric sentinel codes 9/99/999 only for known binary/ordinal columns
    # (be careful -- 9 is a valid value for continuous variables like age)

print("Missing value codes standardised.")

# Special: for Clavien columns, replace 9 (Onbekend) with NaN
clav_cols = [c for c in df.columns if c.startswith('clav')]
for c in clav_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce').replace({9: np.nan})
print(f"Clavien columns ({len(clav_cols)}): sentinel 9 replaced with NaN")


---
## 3. Drop Low-Quality Variables

Based on the readiness analysis, we drop:

**A) anamok1-7 (73% missing)** -- the umbrella variable `anamok` is kept.
The subtypes are only recorded when `anamok=1`, leaving everyone else blank.

**B) Very low prevalence comorbidities (<2%)** -- too few positive cases
to provide reliable signal in the model:
`comhiv` (0.6%), `compancr` (0.3%), `comparalys` (0.3%), `comdem` (0.5%)

**C) comlever** -- 33% missing AND only 1.0% prevalence -- double issue.

**D) combind, scorecm** -- borderline low prevalence (1.2%, 1.1%).
We drop these as well to keep the model parsimonious.


In [ ]:
# A) Drop anamok subtypes (73% missing)
anamok_subtypes = [f'anamok{i}' for i in range(1, 8)]
anamok_to_drop = [c for c in anamok_subtypes if c in df.columns]
df = df.drop(columns=anamok_to_drop)
print(f"Dropped anamok subtypes: {anamok_to_drop}")

# B) Drop very low prevalence comorbidities
low_prev = ['comhiv', 'compancr', 'comparalys', 'comdem', 'comlever', 'combind']
low_prev_present = [c for c in low_prev if c in df.columns]
df = df.drop(columns=low_prev_present)
print(f"Dropped low prevalence comorbidities: {low_prev_present}")

# C) Drop scorecm (1.1% -- almost all M0, very few M1 patients)
if 'scorecm' in df.columns:
    df = df.drop(columns=['scorecm'])
    print("Dropped scorecm (1.1% M1 prevalence)")

print(f"Dataset after dropping: {df.shape[0]} rows, {df.shape[1]} columns")


---
## 4. Encode Special Variables

### 4.1 Roken (Smoking Status)

`roken` is stored as Dutch text with 3 meaningful categories:
- `Nooit gerookt` = Never smoked
- `Gestopt` = Former smoker
- `Momenteel` = Current smoker

This is an **ordinal** variable -- Never < Former < Current reflects increasing
cumulative tobacco exposure and risk. We encode it as 0, 1, 2 respectively.


In [ ]:
# Check current values
print("roken value counts before encoding:")
print(df['roken'].value_counts(dropna=False))
print()


In [ ]:
# Encode roken as ordinal (0=Never, 1=Former, 2=Current)
if 'roken' in df.columns:
    # Step 1: force string, strip whitespace, standardise
    roken_clean = df['roken'].astype(str).str.strip()

    # Step 2: replace all known missing codes with NaN marker
    missing_codes = ['##USER_MISSING_99##', 'Onbekend', 'onbekend',
                     'nan', 'NaN', 'None', 'NA', '-', '']
    roken_clean = roken_clean.replace(missing_codes, np.nan)

    # Step 3: map text to ordinal
    roken_map = {
        'Nooit gerookt': 0,
        'Gestopt':       1,
        'Momenteel':     2
    }
    df['roken'] = roken_clean.map(roken_map)  # anything not in map becomes NaN

    n_encoded = df['roken'].notna().sum()
    n_missing = df['roken'].isna().sum()
    print(f"roken encoded successfully: {n_encoded} valid, {n_missing} missing (NaN)")
    print(df['roken'].value_counts(dropna=False))

    if n_encoded == 0:
        print("WARNING: roken encoding produced all NaN -- check raw values above")
else:
    print("WARNING: roken column not found in dataframe")


### 4.2 Other Text-Coded Variables

Check for other columns that may still contain text values
that should be numeric (e.g. Ja/Nee binary columns).


In [ ]:
# Convert Ja/Nee binary columns to 1/0
# roken is excluded -- already encoded as ordinal 0/1/2 in step 4.1
SKIP_JA_NEE = ['roken']

ja_nee_map = {'Ja': 1, 'ja': 1, 'Nee': 0, 'nee': 0,
              'Yes': 1, 'yes': 1, 'No': 0, 'no': 0}

converted = []
for col in df.columns:
    if col in SKIP_JA_NEE:
        continue
    if df[col].dtype == object:
        unique_vals = set(df[col].dropna().unique())
        # Only convert if all non-null values are Ja/Nee variants
        if unique_vals.issubset(set(ja_nee_map.keys())):
            df[col] = df[col].map(ja_nee_map)
            converted.append(col)

if converted:
    print(f"Converted {len(converted)} Ja/Nee columns to 1/0:")
    for c in converted:
        print(f"  {c}")
else:
    print("No Ja/Nee columns found to convert.")

# Show remaining text columns
text_cols = [c for c in df.columns if df[c].dtype == object]
print(f"Remaining text/object columns ({len(text_cols)}):")
print(text_cols[:30])


---
## 5. Encode Categorical Variables

**Ordinal variables** (natural ordering — encode as integers):
- `scorect`: T1=1, T2=2, T3=3, T4=4
- `scorecn`: N0=0, N1=1, N2=2, N3=3
- `asascore`: 1–5 (already numeric)
- `roken`: already encoded in step 4

**Nominal variables** (no natural ordering — one-hot encode):
- `tumlok1`, `tumhist`, `tumkeus`
- `neoadjtype`, `typok`, `aardok`, `procok`, `recontype`, `analig`

Categories with fewer than 20 patients are collapsed into "other" to avoid sparse dummy variables.

**Scaling note:**
Continuous variables are **standardized** (zero mean, unit variance) for LASSO logistic regression, but left **unscaled** for Random Forest and XGBoost — tree-based methods are invariant to monotone transformations of the input features.


In [ ]:
# Force numeric conversion for ordinal columns
ordinal_cols = ['scorect', 'scorecn', 'asascore', 'roken',
                'duur_operatie', 'bloedverlies', 'neoadjaf',
                'age_at_op', 'gewverl', 'lengte', 'gewicht', 'BMI',
                'packyears', 'fev1vc_percentage']

for col in ordinal_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# scorect/scorecn use sentinel codes (9, 99) for unknown -- clean to NaN
# before any downstream aggregation/imputation uses these columns.
for col in ['scorect', 'scorecn']:
    if col in df.columns:
        df[col] = df[col].replace({9: np.nan, 99: np.nan})

print("Ordinal/continuous columns converted to numeric.")

# Check scorect and scorecn distributions
for col in ['scorect', 'scorecn']:
    if col in df.columns:
        print(f"{col} distribution:")
        print(df[col].value_counts(dropna=False).sort_index())


In [ ]:
# One-hot encode nominal categorical variables
NOMINAL_COLS = ['tumlok1', 'tumhist', 'tumkeus', 'neoadjtype',
                'typok', 'aardok', 'procok', 'recontype', 'analig']

nominal_present = [c for c in NOMINAL_COLS if c in df.columns]

dummies_list = []
for col in nominal_present:
    vc = df[col].value_counts()
    # Keep only categories with >= 20 patients (drop very rare categories)
    valid_cats = vc[vc >= 20].index
    col_filtered = df[col].where(df[col].isin(valid_cats), other=np.nan)
    dummies = pd.get_dummies(col_filtered, prefix=col, drop_first=False, dummy_na=False)
    dummies_list.append(dummies)
    print(f"{col}: {dummies.shape[1]} dummy columns created from {len(valid_cats)} categories")

if dummies_list:
    df = pd.concat([df.drop(columns=nominal_present)] + dummies_list, axis=1)

print(f"Dataset after one-hot encoding: {df.shape[0]} rows, {df.shape[1]} columns")


---
## 6. Impute Missing Values

**Strategy:**
- **Continuous variables**: impute with the **median** (robust to outliers)
- **Binary variables**: impute with the **mode** (most frequent value, usually 0)
- **Ordinal variables**: impute with the **median**

We do NOT impute the target variables (`compl`, `derived_clavien_bin`, `comp*` columns).

All imputation is done on the training set in a real model pipeline
(to avoid leakage from test set statistics). Here we impute on the full dataset
for the purpose of the descriptive analysis and baseline modeling.


In [ ]:
# Define final predictor columns
FINAL_PREDICTORS = [
    # Demographics
    'age_at_op', 'geslacht',
    # Fitness
    'asascore', 'fev1vc_percentage',
    # Nutrition
    'gewverl', 'lengte', 'gewicht', 'BMI', 'dietist',
    # Comorbidities
    'comhartfaal', 'comaf', 'comcarritme', 'comlong', 'comperif',
    'comcva', 'commalig', 'commyo', 'comdiam', 'comnier', 'comgiulcus',
    # Lifestyle
    'roken', 'packyears',
    # Prior surgery
    'anamok',
    # Tumor
    'scorect', 'scorecn',
    # Neoadjuvant
    'neoadj', 'neoadjaf', 'neoadjsch',
    # Surgical
    'conversie', 'duur_operatie', 'bloedverlies', 'resadd',
    'curaok', 'salvok', 'reslmm', 'emr',
]

# Add dummy columns that were created from nominal encoding
dummy_prefixes = ['tumlok1_', 'tumhist_', 'tumkeus_', 'neoadjtype_',
                  'typok_', 'aardok_', 'procok_', 'recontype_', 'analig_']
dummy_cols = [c for c in df.columns if any(c.startswith(p) for p in dummy_prefixes)]
FINAL_PREDICTORS += dummy_cols

# Keep only columns that exist in df
FINAL_PREDICTORS = [c for c in FINAL_PREDICTORS if c in df.columns]
print(f"Final predictor count: {len(FINAL_PREDICTORS)}")


In [ ]:
# Separate continuous and binary/ordinal columns
cont_cols = [c for c in FINAL_PREDICTORS
             if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() > 10]
cat_cols  = [c for c in FINAL_PREDICTORS
             if c not in cont_cols]

print(f"Continuous predictors: {len(cont_cols)}")
print(f"Binary/ordinal predictors: {len(cat_cols)}")

# Show missing counts before imputation
missing_before = df[FINAL_PREDICTORS].isna().sum()
missing_before = missing_before[missing_before > 0].sort_values(ascending=False)
print(f"Variables with missing values before imputation ({len(missing_before)}):")
print(missing_before)


In [ ]:
# ── Packyears: handled separately based on smoking status ──
# packyears uses sentinel codes (999, -96, -99, 9) for unknown -- clean to NaN
# BEFORE computing any median, otherwise the sentinel poisons the imputed value.
df['packyears'] = df['packyears'].replace({999: np.nan, -96: np.nan, -99: np.nan, 9: np.nan})

# Never smokers (roken=0) -> packyears = 0 (no tobacco exposure by definition)
df.loc[df['roken'] == 0, 'packyears'] = 0

# Former/current smokers with missing packyears -> median of smokers only
smoker_median = df.loc[df['roken'].isin([1, 2]), 'packyears'].median()
df.loc[df['roken'].isin([1, 2]) & df['packyears'].isna(), 'packyears'] = smoker_median

# Unknown smoking status -> leave as NaN (we do not know their exposure)
print(f"Packyears -- smoker median used for imputation: {smoker_median:.1f}")
print("\nPackyears by smoking group after fix:")
print(df.groupby('roken', dropna=False)['packyears'].agg(['count','median','min','max']))


In [ ]:
# ── General imputation (skip packyears -- already handled above) ──
# roken is NOT skipped here: Section 6 documents ordinal variables as imputed
# with the median/mode, so roken (ordinal, mode-imputed like other
# binary/ordinal predictors) must actually go through this step.
SKIP_IMPUTE = ['packyears']

# Continuous: median
for col in cont_cols:
    if col in SKIP_IMPUTE:
        continue
    if df[col].isna().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

# Binary/ordinal/categorical: mode
for col in cat_cols:
    if col in SKIP_IMPUTE:
        continue
    if df[col].isna().sum() > 0:
        mode_val = df[col].mode()
        if len(mode_val) > 0:
            df[col] = df[col].fillna(mode_val[0])

# Verify
missing_after = df[FINAL_PREDICTORS].isna().sum()
remaining = missing_after[missing_after > 0]
if len(remaining) == 0:
    print("All predictors complete -- no missing values remain.")
else:
    print(f"Missing values remaining ({len(remaining)} variables):")
    print(remaining)


---
## 7. Log-Transform Skewed Continuous Variables

**Which variables?** `bloedverlies` (blood loss), `duur_operatie` (operation duration), `gewverl` (preoperative weight loss).

**Why log-transform?**
These variables are heavily right-skewed: most patients cluster near the median, but extreme outliers exist (e.g., >1000 mL blood loss, >600 min operations). Without transformation:
- LASSO coefficients are pulled by outliers, destabilising the regularisation path
- Tree-based models are less sensitive to scale, but log compression reduces the number of redundant split points and improves training efficiency

**Why `log1p` (i.e., log(x + 1))?**
`log1p` maps zero safely to zero (avoids log(0) = −∞) and is numerically stable. For values >> 1 it is indistinguishable from log(x).

Skewness before and after is reported to confirm the transformation is effective.


In [ ]:
from scipy.stats import skew

LOG_COLS = ['bloedverlies', 'duur_operatie', 'gewverl']
log_present = [c for c in LOG_COLS if c in df.columns]

fig, axes = plt.subplots(2, len(log_present), figsize=(14, 8))

for i, col in enumerate(log_present):
    original = df[col].dropna()
    transformed = np.log1p(original.clip(lower=0))

    # Before
    axes[0, i].hist(original, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0, i].set_title(f'{col} (original)skewness={skew(original):.2f}', fontweight='bold')
    axes[0, i].set_xlabel('Value')

    # After
    axes[1, i].hist(transformed, bins=40, color='seagreen', edgecolor='white', alpha=0.8)
    axes[1, i].set_title(f'{col} (log1p)skewness={skew(transformed):.2f}', fontweight='bold')
    axes[1, i].set_xlabel('log1p(Value)')

plt.suptitle('Before vs After Log Transformation', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Apply log1p transformation
for col in log_present:
    df[col] = np.log1p(df[col].clip(lower=0))
    print(f"log1p applied to {col}")

print("Log transformation complete.")


---
## 8. Final Predictor Overview

Let us take a final look at the clean predictor set before modeling.


In [ ]:
# Final summary of predictors
summary = pd.DataFrame({
    'Variable': FINAL_PREDICTORS,
    'Type': ['continuous' if c in cont_cols else 'binary/ordinal' for c in FINAL_PREDICTORS],
    'N present': [df[c].notna().sum() for c in FINAL_PREDICTORS],
    'N missing': [df[c].isna().sum() for c in FINAL_PREDICTORS],
    'Mean/Mode': [
        round(df[c].mean(), 3) if pd.api.types.is_numeric_dtype(df[c]) else df[c].mode()[0]
        for c in FINAL_PREDICTORS
    ]
})

print(f"Total predictors: {len(FINAL_PREDICTORS)}")
print(f"Continuous: {len(cont_cols)}")
print(f"Binary/ordinal: {len([c for c in FINAL_PREDICTORS if c not in cont_cols])}")
print()
print(summary.to_string(index=False))


---
## 9. Prepare Target Variables

We derive the outcome variables that will be used for modeling.


In [ ]:
# Derive Clavien-Dindo severity
clav_cols = [c for c in df.columns if c.startswith('clav') and c in df.columns]
if clav_cols:
    df['derived_clavien'] = df[clav_cols].apply(
        pd.to_numeric, errors='coerce').replace({9: np.nan}).max(axis=1)
    df['derived_clavien_bin'] = np.where(
        df['derived_clavien'].between(1, 2), 0,
        np.where(df['derived_clavien'].between(3, 7), 1, np.nan))

# Outcome summary
outcomes = ['compl', 'derived_clavien_bin', 'comppulm', 'compcar',
            'compinfect', 'compwond', 'compchylek', 'compand',
            'compgas', 'compuro', 'compthrom', 'compneu', 'opnameduur', 'icdg']
outcomes_present = [c for c in outcomes if c in df.columns]

print("Outcome variable summary:")
print(f"{'Outcome':<25} {'N events':>10} {'% events':>10}")
print("-" * 47)
for col in outcomes_present:
    col_numeric = pd.to_numeric(df[col], errors='coerce')
    # "% events" only makes sense for true binary 0/1 outcome columns --
    # for continuous (e.g. opnameduur) or multi-category (e.g. icdg) columns
    # a mean*100 is not an event rate, so we skip the percentage there.
    is_binary = set(col_numeric.dropna().unique()).issubset({0, 1})
    n = col_numeric.sum()
    if is_binary:
        pct = col_numeric.mean() * 100
        print(f"{col:<25} {int(n):>10} {pct:>9.1f}%")
    else:
        print(f"{col:<25} {n:>10.1f} {'n/a (not binary)':>16}")


---
## 10. Export Clean Dataset

We save two files:
1. **`df_model.csv`** -- full clean dataset with all columns (predictors + outcomes + IDs)
2. **`df_X.csv`** -- predictor matrix only (ready to pass to sklearn)
3. **`df_y.csv`** -- target variables only


In [ ]:
# Build final model dataframe
keep_cols = ['upn'] + FINAL_PREDICTORS + [c for c in outcomes_present if c in df.columns]
keep_cols = [c for c in keep_cols if c in df.columns]
df_model = df[keep_cols].copy()

# X and y
df_X = df_model[FINAL_PREDICTORS]
df_y = df_model[[c for c in outcomes_present if c in df_model.columns]]

# Export
df_model.to_csv('df_model.csv', index=False)
df_X.to_csv('df_X.csv', index=False)
df_y.to_csv('df_y.csv', index=False)

print(f"Exported df_model.csv  -- {df_model.shape[0]} rows, {df_model.shape[1]} columns")
print(f"Exported df_X.csv      -- {df_X.shape[0]} rows, {df_X.shape[1]} predictors")
print(f"Exported df_y.csv      -- {df_y.shape[0]} rows, {df_y.shape[1]} outcomes")
print()
print("Ready for modeling.")


---
## Summary

The preprocessing pipeline completed the following steps:

| Step | Action |
|---|---|
| Missing codes | Replaced ##USER_MISSING_99##, Onbekend, sentinel values with NaN |
| Dropped variables | anamok1-7 (73% missing), comhiv, compancr, comparalys, comdem, comlever, combind, scorecm (low prevalence) |
| roken | Encoded as ordinal: Never=0, Former=1, Current=2 |
| Ja/Nee columns | Converted to 1/0 |
| Nominal encoding | One-hot encoded tumlok1, tumhist, tumkeus, neoadjtype, typok, aardok, procok, recontype, analig |
| Imputation | Median for continuous, mode for binary/ordinal |
| Log-transform | Applied log1p to bloedverlies, duur_operatie, gewverl |
| Export | df_model.csv, df_X.csv, df_y.csv |

**Next step:** Load `df_X.csv` and `df_y.csv` into your modeling notebook and build the baseline models.
